### Imports

In [5]:
import pandas as pd
import matplotlib.pyplot as plt
import sys
from pathlib import Path

ROOT = next(
    (parent for parent in (Path.cwd(), *Path.cwd().parents)
     if (parent / "src" / "paths.py").is_file()),
    None,
)
if ROOT is None:
    raise FileNotFoundError("Could not locate project root containing src/paths.py")
sys.path.insert(0, str(ROOT))

from src.paths import *
from src.nasa_power import download_nasa_power

### Load solar plants

In [6]:
solar_plants_path = GIS / "solar_power_plants.csv"
if solar_plants_path.is_file():
    solar_plants = pd.read_csv(solar_plants_path)
    display(solar_plants)
else:
    print(f"Solar plant input is missing: {solar_plants_path}")
    print(f"Fill datasets/templates/solar_plants_template.csv and save the completed file as {solar_plants_path.name} in {GIS}.")
    solar_plants = pd.DataFrame(columns=["plant_id", "latitude", "longitude"])

Solar plant input is missing: D:\My_Project\Analyse\4G-Network-Coverage-Optimization\datasets\raw\gis\solar_power_plants.csv
Fill datasets/templates/solar_plants_template.csv and save the completed file as solar_power_plants.csv in D:\My_Project\Analyse\4G-Network-Coverage-Optimization\datasets\raw\gis.


### Download NASA POWER data

In [7]:
START_DATE = "20240101"
END_DATE = "20241231"

if solar_plants.empty:
    print("NASA POWER download skipped: no populated solar plant coordinates are available.")

for _, plant in solar_plants.iterrows():

    plant_id = str(plant["plant_id"])

    output_file = (
        NASA / f"{plant_id}_hourly.csv"
    )

    if output_file.exists():
        print("Already downloaded:", plant_id)
        continue

    df = download_nasa_power(
        latitude=float(plant["latitude"]),
        longitude=float(plant["longitude"]),
        start_date=START_DATE,
        end_date=END_DATE,
        output_path=output_file
    )

NASA POWER download skipped: no populated solar plant coordinates are available.


### Inspect solar irradiance

In [8]:
files = sorted([f for f in NASA.glob("*.csv") if f.is_file()])
if not files:
    print(f"No NASA hourly files found in {NASA}. Populate the solar plant input and run the download cell first.")


for file in files:

    df = pd.read_csv(file)

    df["timestamp"] = pd.to_datetime(df["timestamp"])

    plt.figure(figsize=(14, 4))

    plt.plot(
        df["timestamp"],
        df["solar_irradiance"]
    )

    plt.title(f"Solar Irradiance – {file.stem}")
    plt.xlabel("Time")
    plt.ylabel("NASA irradiance value (check units)")
    plt.tight_layout()


No NASA hourly files found in D:\My_Project\Analyse\4G-Network-Coverage-Optimization\datasets\raw\solar\nasa_power_hourly. Populate the solar plant input and run the download cell first.


### Daily solar resource analysis

In [9]:
for file in files:

    df = pd.read_csv(file)

    df["timestamp"] = pd.to_datetime(df["timestamp"])

    df["date"] = df["timestamp"].dt.date

    daily = df.groupby("date")[
        "solar_irradiance"
    ].mean()

    daily.plot(
        figsize=(14, 4),
        title=f"Daily Mean Solar Resource – {file.stem}"
    )

    plt.ylabel("Mean hourly irradiance value")
    plt.tight_layout()
    plt.show()